# YOLO26 Video Inference

Runs a YOLO26 `.pt` model on every frame of an MP4 that is already captured at 1 fps.

**Upload your `.pt` model and `.mp4` video when prompted.**

## Step 1: Install & import

In [ ]:
!pip install -q ultralytics

import cv2
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

## Step 2: Upload model and video

In [ ]:
print('Upload your YOLO26 .pt model file:')
model_upload = files.upload()
model_path = next(k for k in model_upload if k.endswith('.pt'))

print('\nUpload your .mp4 video (already at 1 fps):')
video_upload = files.upload()
video_path = next(k for k in video_upload if k.lower().endswith('.mp4'))

print(f'\nModel: {model_path}')
print(f'Video: {video_path}')

## Step 3: Upload IMU/GPS and video sync CSVs

- **IMU/GPS CSV**: your `trail_data.csv` (columns with `ms`, gravity vector, GPS, etc.).
- **Video sync CSV**: a single value — the millisecond timestamp of the first video frame. The first numeric cell is used, so column name doesn't matter.

In [ ]:
print('Upload your IMU/GPS CSV (ms + gravity + GPS):')
imu_upload = files.upload()
imu_csv = next(k for k in imu_upload if k.lower().endswith('.csv'))
imu_df = pd.read_csv(imu_csv)
print(f'  loaded {len(imu_df)} rows, columns: {list(imu_df.columns)}')

print('\nUpload your video sync CSV (ms of the first video frame):')
sync_upload = files.upload()
sync_csv = next(k for k in sync_upload if k.lower().endswith('.csv'))
sync_df = pd.read_csv(sync_csv)

# Grab the first numeric value as the video start offset, tolerant of column naming.
video_start_ms = int(sync_df.select_dtypes('number').iloc[0, 0])
print(f'\nvideo_start_ms = {video_start_ms}')

## Step 4: Run tracking on every frame

Since the video is already at 1 fps, we process every frame (no sub-sampling).
Uses `model.track(..., persist=True)` with ByteTrack so each object keeps a
stable `track_id` across frames (masks too, if this is a `-seg` model).
Annotated frames are written to `output.mp4` and per-detection rows
(including `mask_area`) to `detections.csv`.

In [ ]:
import numpy as np
import torch

model = YOLO(model_path)

# --- Tunables ----------------------------------------------------------------
# Per-class confidence thresholds. Classes not listed use DEFAULT_CONF.
# Keys are matched case-insensitively against model.names.
CLASS_CONF = {
    'bench': 0.15,
    'rough trail': 0.15,
}
DEFAULT_CONF = 0.35

# Overlap suppression: drop `weaker` whenever it overlaps `stronger`.
SUPPRESS_RULES = [
    ('crushed gravel trail', 'rough trail'),  # rough trail always wins
]
IOU_SUPPRESS = 0.3  # mask-IoU; lower toward 0 for looser "any overlap" behavior

# FP16 inference if a CUDA GPU is available (ignored on CPU).
USE_HALF = torch.cuda.is_available()
# -----------------------------------------------------------------------------

print('Model class names:', model.names)
print(f'FP16 inference: {USE_HALF}')
name_to_id = {v.lower(): k for k, v in model.names.items()}

def conf_for(class_id):
    return CLASS_CONF.get(model.names.get(class_id, '').lower(), DEFAULT_CONF)

def iou_mask(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

GLOBAL_CONF = min([DEFAULT_CONF] + list(CLASS_CONF.values()))

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {width}x{height}, {fps:.2f} fps, {total} frames')

out_video = 'output.mp4'
writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

rows = []
frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model.track(frame, persist=True, tracker='bytetrack.yaml',
                          conf=GLOBAL_CONF, half=USE_HALF, verbose=False)[0]

    if results.boxes is not None and len(results.boxes) > 0:
        confs = results.boxes.conf.cpu().numpy()
        clss = results.boxes.cls.cpu().numpy().astype(int)
        masks_np = (results.masks.data.cpu().numpy().astype(bool)
                    if results.masks is not None else None)

        # 1. Per-class confidence filter
        keep = [i for i, (c, k) in enumerate(zip(confs, clss)) if c >= conf_for(int(k))]

        # 2. Overlap suppression using mask IoU (drop weaker when overlapping stronger)
        suppressed = set()
        if masks_np is not None:
            for weaker_name, stronger_name in SUPPRESS_RULES:
                w_id = name_to_id.get(weaker_name.lower())
                s_id = name_to_id.get(stronger_name.lower())
                if w_id is None or s_id is None:
                    continue
                for i in keep:
                    if int(clss[i]) != w_id:
                        continue
                    for j in keep:
                        if int(clss[j]) != s_id:
                            continue
                        if iou_mask(masks_np[i], masks_np[j]) > IOU_SUPPRESS:
                            suppressed.add(i)
                            break
        keep = [i for i in keep if i not in suppressed]

        # Filter Results in place so plot() and CSV both see only survivors
        mask_t = torch.zeros(len(results.boxes), dtype=torch.bool,
                             device=results.boxes.data.device)
        if keep:
            mask_t[keep] = True
        results.boxes = results.boxes[mask_t]
        if results.masks is not None:
            results.masks = results.masks[mask_t]

    annotated = results.plot()
    writer.write(annotated)

    if results.boxes is not None and len(results.boxes) > 0:
        boxes = results.boxes.xyxy.cpu().numpy()
        confs = results.boxes.conf.cpu().numpy()
        clss = results.boxes.cls.cpu().numpy().astype(int)
        ids = (results.boxes.id.cpu().numpy().astype(int)
               if results.boxes.id is not None
               else [-1] * len(boxes))
        if results.masks is not None:
            areas = results.masks.data.cpu().numpy().astype(bool).sum(axis=(1, 2))
        else:
            areas = [0] * len(boxes)
        for (x1, y1, x2, y2), conf, cls, tid, area in zip(boxes, confs, clss, ids, areas):
            rows.append({
                'frame': frame_idx,
                't_sec': frame_idx,
                'track_id': int(tid),
                'class_id': int(cls),
                'class_name': model.names.get(int(cls), str(cls)),
                'conf': float(conf),
                'mask_area': int(area),
                'x1': float(x1), 'y1': float(y1),
                'x2': float(x2), 'y2': float(y2),
            })

    frame_idx += 1
    if frame_idx % 25 == 0:
        print(f'  processed {frame_idx}/{total} frames')

cap.release()
writer.release()

det = pd.DataFrame(rows)
det.to_csv('detections.csv', index=False)
n_tracks = det['track_id'].nunique() if len(det) else 0
print(f'\nDone. {frame_idx} frames processed, {len(det)} detections, {n_tracks} unique tracks.')
det.head()

## Step 5: Build segments + points CSV with GPS, extract reference frames

- **Trail classes** (`rough trail`, `crushed gravel trail`, `wood trail`) — per frame pick the
  trail with the largest mask area, smooth with a confidence-weighted majority vote over
  `SMOOTH_WINDOW` frames, and split into segments. Segments shorter than `MIN_SEG_FRAMES` are
  dropped as noise.
- **Point classes** (`bench`, `sign`) — group consecutive detections; any gap longer than
  `POINT_GAP_RESET` frames starts a new instance. Each instance uses the **last** frame of
  its group as the reference.
- Each entry is tagged with GPS (nearest-ms lookup in `imu_df`) and a reference frame is
  saved to `frames/` for mapping.

In [ ]:
import os, shutil
from collections import Counter

# --- Tunables ----------------------------------------------------------------
TRAIL_CLASSES  = ['rough trail', 'crushed gravel trail', 'wood trail']
POINT_CLASSES  = ['bench', 'sign']
MIN_SEG_FRAMES = 3    # drop trail segments shorter than this (noise)
SMOOTH_WINDOW  = 5    # conf-weighted majority vote window (centered, in frames)
POINT_GAP_RESET = 10  # frames without a point detection -> new instance
# -----------------------------------------------------------------------------

trail_names_lower = {c.lower() for c in TRAIL_CLASSES}

# Total frame count (re-query so we don't depend on tracking cell state)
_tmp = cv2.VideoCapture(video_path)
total_frames = int(_tmp.get(cv2.CAP_PROP_FRAME_COUNT))
_tmp.release()

# GPS lookup helpers -- keep only rows with valid GPS
gps_df = imu_df.dropna(subset=['lat', 'lng']).reset_index(drop=True)

def frame_to_ms(f):
    return int(video_start_ms + f * 1000)

def nearest_gps(ms):
    if gps_df.empty:
        return None, None
    idx = (gps_df['ms'] - ms).abs().idxmin()
    return float(gps_df.loc[idx, 'lat']), float(gps_df.loc[idx, 'lng'])

# --- 1) Per-frame dominant trail class (largest mask area) -------------------
trail_dets = det[det['class_name'].str.lower().isin(trail_names_lower)]
per_frame = {}
for _, row in trail_dets.iterrows():
    f = int(row['frame'])
    if f not in per_frame or row['mask_area'] > per_frame[f]['area']:
        per_frame[f] = {'class': row['class_name'],
                        'conf': float(row['conf']),
                        'area': float(row['mask_area'])}

# --- 2) Conf-weighted majority vote over a centered window -------------------
half_w = SMOOTH_WINDOW // 2
def smoothed_label(center):
    votes = Counter()
    for f in range(max(0, center - half_w), min(total_frames, center + half_w + 1)):
        info = per_frame.get(f)
        if info:
            votes[info['class']] += info['conf']
    return votes.most_common(1)[0][0] if votes else None

smoothed = [smoothed_label(f) for f in range(total_frames)]

# --- 3) Run-length encode into segments --------------------------------------
segments = []
start, current = None, None
for f, lbl in enumerate(smoothed):
    if lbl != current:
        if current is not None:
            segments.append({'class': current, 'start_frame': start, 'end_frame': f - 1})
        current = lbl
        start = f
if current is not None:
    segments.append({'class': current, 'start_frame': start, 'end_frame': total_frames - 1})

# Drop short noise segments
segments = [s for s in segments if (s['end_frame'] - s['start_frame'] + 1) >= MIN_SEG_FRAMES]

# Merge adjacent segments of the same class (can happen after short-segment removal)
merged = []
for s in segments:
    if merged and merged[-1]['class'] == s['class']:
        merged[-1]['end_frame'] = s['end_frame']
    else:
        merged.append(s)
segments = merged

# --- 4) Build segment + point rows -------------------------------------------
rows_out = []

for s in segments:
    sf, ef = s['start_frame'], s['end_frame']
    start_ms, end_ms = frame_to_ms(sf), frame_to_ms(ef)
    slat, slng = nearest_gps(start_ms)
    elat, elng = nearest_gps(end_ms)
    mid = (sf + ef) // 2
    seg_confs = det[(det['frame'].between(sf, ef)) &
                    (det['class_name'].str.lower() == s['class'].lower())]['conf']
    rows_out.append({
        'type': 'segment',
        'class': s['class'],
        'start_frame': sf, 'end_frame': ef, 'rep_frame': mid,
        'n_frames': ef - sf + 1,
        'start_ms': start_ms, 'end_ms': end_ms,
        'start_lat': slat, 'start_lng': slng,
        'end_lat': elat, 'end_lng': elng,
        'mean_conf': float(seg_confs.mean()) if len(seg_confs) else None,
    })

for cls in POINT_CLASSES:
    cls_dets = det[det['class_name'].str.lower() == cls.lower()].sort_values('frame')
    if cls_dets.empty:
        continue
    groups = []
    current_group = [cls_dets.iloc[0]]
    for _, row in cls_dets.iloc[1:].iterrows():
        if row['frame'] - current_group[-1]['frame'] > POINT_GAP_RESET:
            groups.append(current_group)
            current_group = [row]
        else:
            current_group.append(row)
    groups.append(current_group)
    for g in groups:
        last = g[-1]
        f = int(last['frame']); ms = frame_to_ms(f)
        lat, lng = nearest_gps(ms)
        rows_out.append({
            'type': 'point',
            'class': cls,
            'start_frame': f, 'end_frame': f, 'rep_frame': f,
            'n_frames': 1,
            'start_ms': ms, 'end_ms': ms,
            'start_lat': lat, 'start_lng': lng,
            'end_lat': lat, 'end_lng': lng,
            'mean_conf': float(last['conf']),
        })

summary = (pd.DataFrame(rows_out)
             .sort_values('rep_frame')
             .reset_index(drop=True))

# --- 5) Extract representative frames and save as jpgs -----------------------
os.makedirs('frames', exist_ok=True)
for f in os.listdir('frames'):
    os.remove(os.path.join('frames', f))

cap = cv2.VideoCapture(video_path)
image_paths = []
for _, row in summary.iterrows():
    f = int(row['rep_frame'])
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, frame = cap.read()
    safe = row['class'].replace(' ', '_')
    path = f'frames/{row["type"]}_{safe}_f{f:05d}.jpg'
    if ok:
        cv2.imwrite(path, frame)
        image_paths.append(path)
    else:
        image_paths.append('')
cap.release()
summary['image'] = image_paths

summary.to_csv('trail_summary.csv', index=False)
if os.path.exists('frames.zip'):
    os.remove('frames.zip')
shutil.make_archive('frames', 'zip', 'frames')

n_seg = (summary['type'] == 'segment').sum() if len(summary) else 0
n_pt  = (summary['type'] == 'point').sum()   if len(summary) else 0
print(f'Segments: {n_seg}')
print(f'Points:   {n_pt}')
print(f'Wrote trail_summary.csv and frames.zip ({len(image_paths)} images)')
summary

## Step 6: Download results

In [ ]:
files.download('output.mp4')
files.download('detections.csv')
files.download('trail_summary.csv')
files.download('frames.zip')

## Step 7: Annotated Excel export (waypoint_photos sheet)

Reads `trail_summary.csv` and `detections.csv` and emits
`trail_data_annotated_generated.xlsx` with a single `waypoint_photos` sheet:
one row per 1-fps waypoint, columns in the dashboard-compatible order
`wp, image_file, surface_type, obstacles, infrastructure, lat, lng, start_ms, end_ms, mean_conf`.

In [ ]:
import numpy as np
import pandas as pd

summary = pd.read_csv('trail_summary.csv') if Path('trail_summary.csv').exists() else pd.DataFrame()
dets    = pd.read_csv('detections.csv')    if Path('detections.csv').exists()    else pd.DataFrame()

OBSTACLE_TOKENS = ['log on ground', 'rock', 'root', 'branch', 'debris', 'hole', 'puddle']
INFRA_TOKENS    = ['sign', 'bench', 'boardwalk', 'railing', 'fence', 'post', 'bridge', 'trail blaze']
VIDEO_START_MS_FALLBACK = 11187  # only used when summary/point row is missing

def matches_tokens(class_name, tokens):
    cn = str(class_name).lower().strip()
    return any(t in cn for t in tokens)

segs = summary[summary['type'] == 'segment'].reset_index(drop=True) if len(summary) else summary
pts  = summary[summary['type'] == 'point'].reset_index(drop=True)   if len(summary) else summary

# Waypoint range spans both CSVs
max_frame = 0
if len(segs):
    max_frame = max(max_frame, int(segs['end_frame'].max()))
if len(pts):
    max_frame = max(max_frame, int(pts['end_frame'].max()))
if len(dets):
    max_frame = max(max_frame, int(dets['frame'].max()))
waypoints = np.arange(0, max_frame + 1)

# Vectorized containment: wp x segment boolean, then argmax by mean_conf
if len(segs):
    wp_col   = waypoints[:, None]
    sf       = segs['start_frame'].to_numpy()[None, :]
    ef       = segs['end_frame'].to_numpy()[None, :]
    conf_vec = segs['mean_conf'].fillna(-np.inf).to_numpy()
    contains = (wp_col >= sf) & (wp_col <= ef)
    masked   = np.where(contains, conf_vec[None, :], -np.inf)
    best     = masked.argmax(axis=1)
    has_seg  = contains.any(axis=1)
    seg_idx_for_wp = np.where(has_seg, best, -1)
else:
    seg_idx_for_wp = np.full(len(waypoints), -1)

# Point lookup by rep_frame (first match wins if duplicates)
point_for_wp = {}
if len(pts):
    for _, p in pts.iterrows():
        f = int(p['rep_frame'])
        point_for_wp.setdefault(f, p)

# Detections grouped by frame (int keys)
if len(dets):
    dets = dets.copy()
    dets['frame'] = dets['frame'].astype(int)
    det_by_frame = {f: g for f, g in dets.groupby('frame')}
else:
    det_by_frame = {}

obstacle_classes_matched = set()
infra_classes_matched    = set()
unmatched_classes        = set()
all_det_classes          = set()

rows_out = []
for pos, wp in enumerate(waypoints):
    wp = int(wp)
    seg_i = int(seg_idx_for_wp[pos])
    seg   = segs.iloc[seg_i] if seg_i >= 0 else None
    frame_dets = det_by_frame.get(wp, pd.DataFrame())
    pt = point_for_wp.get(wp)

    has_dets    = len(frame_dets) > 0
    in_segment  = seg is not None
    image_file  = f'img_{wp:04d}.jpg' if (has_dets or in_segment) else ''

    surface_type = seg['class'] if in_segment else 'No image linked in segments'

    obstacles_list, infra_list = [], []
    obs_tids, obs_names = set(), set()
    inf_tids, inf_names = set(), set()

    for _, d in frame_dets.iterrows():
        cn = str(d['class_name'])
        all_det_classes.add(cn)
        is_obs = matches_tokens(cn, OBSTACLE_TOKENS)
        is_inf = matches_tokens(cn, INFRA_TOKENS)

        try:
            tid = int(d['track_id'])
        except (ValueError, TypeError):
            tid = -1

        if is_obs:
            obstacle_classes_matched.add(cn)
            key_set = obs_tids if tid >= 0 else obs_names
            key     = tid      if tid >= 0 else cn.lower().strip()
            if key not in key_set:
                key_set.add(key)
                obstacles_list.append(cn)
        if is_inf:
            infra_classes_matched.add(cn)
            key_set = inf_tids if tid >= 0 else inf_names
            key     = tid      if tid >= 0 else cn.lower().strip()
            if key not in key_set:
                key_set.add(key)
                infra_list.append(cn)
        if not is_obs and not is_inf:
            unmatched_classes.add(cn)

    obstacles      = '; '.join(obstacles_list) if obstacles_list else 'None detected'
    infrastructure = '; '.join(infra_list)     if infra_list     else 'None detected'

    # lat / lng: segment -> point -> NaN
    lat = lng = np.nan
    if in_segment and pd.notna(seg.get('start_lat')):
        lat, lng = float(seg['start_lat']), float(seg['start_lng'])
    elif pt is not None and pd.notna(pt.get('start_lat')):
        lat, lng = float(pt['start_lat']), float(pt['start_lng'])

    # start_ms / end_ms: segment -> point -> synth
    if in_segment:
        start_ms = int(seg['start_ms'])
        end_ms   = int(seg['end_ms'])
    elif pt is not None:
        start_ms = int(pt['start_ms'])
        end_ms   = int(pt['end_ms'])
    else:
        start_ms = wp * 1000 + VIDEO_START_MS_FALLBACK
        end_ms   = start_ms + 1000

    # mean_conf
    if in_segment and pd.notna(seg.get('mean_conf')):
        mean_conf = float(seg['mean_conf'])
    elif has_dets:
        mean_conf = float(frame_dets['conf'].mean())
    else:
        mean_conf = np.nan

    rows_out.append({
        'wp': wp,
        'image_file': image_file,
        'surface_type': surface_type,
        'obstacles': obstacles,
        'infrastructure': infrastructure,
        'lat': lat,
        'lng': lng,
        'start_ms': start_ms,
        'end_ms': end_ms,
        'mean_conf': mean_conf,
    })

cols = ['wp', 'image_file', 'surface_type', 'obstacles', 'infrastructure',
        'lat', 'lng', 'start_ms', 'end_ms', 'mean_conf']
out_df = pd.DataFrame(rows_out, columns=cols)

output_path = 'trail_data_annotated_generated.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as w:
    out_df.to_excel(w, sheet_name='waypoint_photos', index=False)

# --- Report ------------------------------------------------------------------
covered   = int((out_df['surface_type'] != 'No image linked in segments').sum())
uncovered = int((out_df['surface_type'] == 'No image linked in segments').sum())

print(f'Output: {output_path}')
print(f'Rows:   {len(out_df)}')
print(f'Waypoints with surface segment:    {covered}')
print(f'Waypoints without surface segment: {uncovered}')
print()
print('Detection class -> bucket:')
if not all_det_classes:
    print('  (no detections)')
for cn in sorted(all_det_classes):
    buckets = []
    if matches_tokens(cn, OBSTACLE_TOKENS):
        buckets.append('obstacle')
    if matches_tokens(cn, INFRA_TOKENS):
        buckets.append('infrastructure')
    if not buckets:
        buckets = ['unmatched']
    print(f'  {cn!r}: {"+".join(buckets)}')

unmatched_only = sorted(unmatched_classes - obstacle_classes_matched - infra_classes_matched)
if unmatched_only:
    print()
    print('Unmatched classes (add to OBSTACLE_TOKENS / INFRA_TOKENS if needed):')
    for cn in unmatched_only:
        print(f'  {cn!r}')

print()
print('Preview:')
out_df.head(10)